In [1]:
import pandas as pd
import glob

# Get all files
files = glob.glob("blast_outputs/*.txt")

all_data = []

for file in files:
    df = pd.read_csv(file, sep=r"\s+", header=None)
    
    # Adjust based on your format (5 columns)
    df.columns = ["qseqid", "sseqid", "pident", "length", "similarity_score"]
    
    all_data.append(df)

# Combine all
combined_df = pd.concat(all_data, ignore_index=True)

# Extract UniProt ID
combined_df["uniprot_id"] = combined_df["qseqid"].apply(lambda x: x.split("|")[1])

# Save final CSV
combined_df.to_csv("data/all_blast_results.csv", index=False)

print("All files merged successfully!")

All files merged successfully!


In [2]:
best_df = combined_df.sort_values("similarity_score", ascending=False) \
                     .groupby("uniprot_id") \
                     .first() \
                     .reset_index()

best_df.to_csv("data/best_templates.csv", index=False)

In [5]:
best_df["method"] = best_df["similarity_score"].apply(
    lambda x: "MODELLER" if x > 35 else "I-TASSER"
)

best_df.to_csv("data/classified_results.csv", index=False)